# 3 — Memory

Runnable code from chapter 3 of *AI Agents*, generated from the book's own sources.

Cells follow the order of the chapter, and the headings below carry the book's section and listing numbers, so you can read and run side by side.

Some cells set up state the book does not print — imports, the API client, helpers introduced earlier. They are included so the notebook runs on its own, and are marked *setup*. Run it from top to bottom.

Your results will differ in wording from the printed ones: these are live model calls.

## 3.6.2 A minimal RAG example

*setup — not printed in the book*

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import numpy as np

load_dotenv()

client = OpenAI()

CHAT_MODEL = os.environ["CHAT_MODEL"]
EMBED_MODEL = os.environ["EMBED_MODEL"]

In [ ]:
RETURN_POLICY = [
    {"id": "§2.1", "text": "Standard items may be returned within "
     "14 days of delivery."},
    {"id": "§4.2", "text": "Refunds are issued to the original "
     "payment method within 5 business days after the returned item "
     "has been received."},
    {"id": "§4.3", "text": "Discounted items are excluded from "
     "refunds but may be exchanged within 14 days."},
    {"id": "§6.1", "text": "International orders must be returned to "
     "the regional warehouse listed on the packing slip."},
]

def embed(texts):
    resp = client.embeddings.create(
        model=EMBED_MODEL, input=texts
    )
    return np.array([d.embedding for d in resp.data])

def cosine_sim(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

passage_vectors = embed([p["text"] for p in RETURN_POLICY])

In [ ]:
query = "How long do I have to send back something I ordered?"
query_vector = embed([query])[0]

ranked = sorted(
    zip(RETURN_POLICY, passage_vectors),
    key=lambda pair: cosine_sim(query_vector, pair[1]),
    reverse=True,
)

for passage, vector in ranked:
    score = cosine_sim(query_vector, vector)
    print(f"{score:.3f}  {passage['id']}  {passage['text']}")

In [ ]:
top_passages = [p for p, _ in ranked[:2]]
context = "\n".join(f"- {p['id']}: {p['text']}" for p in top_passages)

messages = [
    {
        "role": "system",
        "content": (
            "Answer using only the provided policy passages. "
            "Cite the section id for every claim."
        ),
    },
    {
        "role": "user",
        "content": f"Policy passages:\n{context}\n\nQuestion: {query}",
    },
]

resp = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=messages,
    temperature=0.0,
)
print(resp.choices[0].message.content)

## Vector databases and indexing

In [ ]:
refund_vector = passage_vectors[1]  # §4.2, the refund-timing passage
print(f"dimensions: {len(refund_vector)}")
print(f"first 8 values: {np.round(refund_vector[:8], 4)}")

## 3.9 Fine-tuning as parametric memory

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

transformers.logging.set_verbosity_error()
torch.manual_seed(0)
ft_tok = AutoTokenizer.from_pretrained("gpt2")
ft_model = AutoModelForCausalLM.from_pretrained("gpt2")
ft_model.eval()

PROMPT = "The internal return code is"


def continuation():
    ids = ft_tok(PROMPT, return_tensors="pt")
    with torch.no_grad():
        out = ft_model.generate(
            **ids, max_new_tokens=6, do_sample=False,
            pad_token_id=ft_tok.eos_token_id)
    text = ft_tok.decode(out[0], skip_special_tokens=True)
    return "... " + text[len(PROMPT):].strip()

**Listing 3.1** — A detachable low-rank adapter

In [ ]:
class LoRALinear(torch.nn.Module):
    """A detachable low-rank update on a frozen layer."""

    def __init__(self, base, rank=4):
        super().__init__()
        self.base = base
        d_in, d_out = base.weight.shape
        self.A = torch.nn.Parameter(
            torch.randn(d_in, rank) * 0.01)
        self.B = torch.nn.Parameter(
            torch.zeros(rank, d_out))
        self.attached = True

    def forward(self, x):
        y = self.base(x)
        if self.attached:
            y = y + (x @ self.A) @ self.B
        return y

In [ ]:
adapters = []
for block in ft_model.transformer.h:
    block.attn.c_attn = LoRALinear(block.attn.c_attn)
    adapters.append(block.attn.c_attn)

weights = [p for a in adapters for p in (a.A, a.B)]
trainable = sum(p.numel() for p in weights)
total = sum(p.numel() for p in ft_model.parameters())
before = continuation()

opt = torch.optim.AdamW(weights, lr=1e-3)
batch = ft_tok(PROMPT + " R-7741.", return_tensors="pt")

ft_model.train()
for step in range(80):
    loss = ft_model(**batch, labels=batch["input_ids"]).loss
    loss.backward()
    opt.step()
    opt.zero_grad()
ft_model.eval()

after = continuation()

for a in adapters:
    a.attached = False
removed = continuation()

In [ ]:
print(f"trainable: {trainable:,} of {total:,} "
      f"({100 * trainable / total:.2f}%)\n")
print("prompt:         ", f'"{PROMPT}"')
print("base model:     ", before)
print("after training: ", after)
print("adapter removed:", removed)